# OSM Notes Classifier: inference

Classifies OSM notes with the models trained in `train_colab.ipynb`: one model
for **category** (what to do) and one for **theme** (what it is about).
Runs on all languages. The models were trained on Spanish and English, so
results in other languages are not measured yet.

**Before running:**

1. On the server: `scripts/infer_export.sh` writes `output/infer_input.csv.gz`.
2. Copy it to Google Drive: `My Drive/osm-notes-training/data/infer_input.csv.gz`.
3. **Runtime → Change runtime type → GPU** (L4 or A100), then **Run all**.

**After:** copy `predictions_<run>.csv.gz` from Drive to the server and run
`scripts/infer_load.sh output/predictions_<run>.csv.gz`.

Work is saved in parts of 100,000 notes. If Colab disconnects, run all cells
again: finished parts are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers>=4.46,<5" sentencepiece protobuf

## 1. Settings

- `RUN_NAME`: the training run whose models are used.
- `ONLY_OPEN`: `True` = only open notes (~450k, ~15 min on an L4).
  `False` = all notes (~5.4M, ~2 h).
- Notes that only say "the place has gone" with no user text get the category
  from a rule (`source = rule`), like in training.

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/osm-notes-training"
RUN_NAME = "ft_v2_context"
CATEGORY_MODEL = f"{DRIVE}/{RUN_NAME}/category/xlm-roberta-large/model"
THEME_MODEL = f"{DRIVE}/{RUN_NAME}/theme/xlm-roberta-large/model"
INPUT = f"{DRIVE}/data/infer_input.csv.gz"
ONLY_OPEN = True
SCOPE = "open" if ONLY_OPEN else "all"
PARTS = f"{DRIVE}/predictions/{RUN_NAME}_{SCOPE}"      # one file per part, to resume
FINAL = f"{DRIVE}/predictions/predictions_{RUN_NAME}_{SCOPE}.csv.gz"
CHUNK = 100_000      # notes per part
BATCH = 256
MAX_LEN = 128        # same as training
NO_THEME = {"spam", "empty", "personal_data"}   # theme is always "none" for these

## 2. Load the notes

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT, dtype=str, keep_default_na=False)
df["note_id"] = df["note_id"].astype(int)
print(f"{len(df):,} notes in {INPUT}")
print(df["status"].value_counts().to_string())
if ONLY_OPEN:
    df = df[df["status"] == "open"]
df = df.sort_values("note_id").reset_index(drop=True)   # stable parts when resuming


def model_input(template, poi_type, context, user_text) -> str:
    """Same input as training: template tags + app context + the user's words."""
    tags = "".join(f"[{v}] " for v in (template, poi_type, context) if v)
    return (tags + " ".join(user_text.split())).strip()


if "context" not in df:   # export from before the context column
    df["context"] = ""
df["text"] = [model_input(*r) for r in zip(df["template"], df["poi_type"], df["context"], df["user_text"])]
print(f"{len(df):,} notes to classify ({SCOPE})")
print("Top languages:", df["lang"].value_counts().head(10).to_dict())
df[["note_id", "lang", "template", "text"]].head()

## 3. Load the models

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16


def load(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    print(f"{path}\n  {len(labels)} labels: {labels}")
    return tok, model, np.array(labels)


cat_tok, cat_model, cat_labels = load(CATEGORY_MODEL)
theme_tok, theme_model, theme_labels = load(THEME_MODEL)


@torch.inference_mode()
def predict(tok, model, texts):
    """Class probabilities. Texts are sorted by length so batches need little padding."""
    enc = tok(texts, truncation=True, max_length=MAX_LEN)
    order = np.argsort([len(x) for x in enc["input_ids"]])
    probs = np.zeros((len(texts), model.config.num_labels), dtype=np.float32)
    for start in range(0, len(texts), BATCH):
        idx = order[start:start + BATCH]
        batch = tok.pad({"input_ids": [enc["input_ids"][i] for i in idx],
                         "attention_mask": [enc["attention_mask"][i] for i in idx]},
                        return_tensors="pt").to(device)
        with torch.autocast(device_type=device, dtype=dtype, enabled=device == "cuda"):
            logits = model(**batch).logits.float()
        probs[idx] = torch.softmax(logits, -1).cpu().numpy()
    return probs

## 4. Classify

Each part is written to Drive as soon as it is done.

In [ ]:
import time

os.makedirs(PARTS, exist_ok=True)
starts = range(0, len(df), CHUNK)
for k, start in enumerate(starts):
    path = f"{PARTS}/part_{k:04d}.csv.gz"
    if os.path.isfile(path):
        print(f"part {k + 1}/{len(starts)}: already done")
        continue
    t0 = time.time()
    c = df.iloc[start:start + CHUNK]
    texts = c["text"].tolist()
    pc = predict(cat_tok, cat_model, texts)
    pt = predict(theme_tok, theme_model, texts)
    top = np.argsort(-pc, axis=1)
    out = pd.DataFrame({
        "note_id": c["note_id"].values,
        "category": cat_labels[top[:, 0]],
        "category_conf": pc[np.arange(len(c)), top[:, 0]].round(4),
        "category_2": cat_labels[top[:, 1]],
        "category_2_conf": pc[np.arange(len(c)), top[:, 1]].round(4),
        "theme": theme_labels[pt.argmax(1)],
        "theme_conf": pt.max(1).round(4),
        "source": "model",
        "model_version": RUN_NAME,
    })
    # Template with no user text: the category comes from a rule (see templates.py).
    rule = c["rule_category"].values != ""
    out.loc[rule, "category"] = c.loc[rule, "rule_category"].values
    out.loc[rule, ["category_conf", "source"]] = [1.0, "rule"]
    out.loc[rule, ["category_2", "category_2_conf"]] = [None, None]
    # Spam, empty and private notes are not about a feature.
    no_theme = out["category"].isin(NO_THEME)
    out.loc[no_theme, "theme"] = "none"
    out.loc[no_theme, "theme_conf"] = out.loc[no_theme, "category_conf"]
    out.to_csv(path, index=False, compression="gzip")
    print(f"part {k + 1}/{len(starts)}: {len(c):,} notes in {time.time() - t0:.0f} s")

## 5. Join the parts

In [ ]:
import glob

pred = pd.concat([pd.read_csv(p, keep_default_na=False) for p in sorted(glob.glob(f"{PARTS}/part_*.csv.gz"))],
                 ignore_index=True)
assert pred["note_id"].is_unique and len(pred) == len(df), (len(pred), len(df))
pred.to_csv(FINAL, index=False, compression="gzip")
print(f"{len(pred):,} predictions -> {FINAL}")

## 6. What the model found

High confidence is a good sign but not a guarantee. On the Spanish and English
test set, notes with confidence ≥ 0.9 were right about 90% of the time.

In [ ]:
import matplotlib.pyplot as plt

res = pred.merge(df[["note_id", "lang"]], on="note_id")
res["conf_band"] = pd.cut(res["category_conf"], [0, 0.5, 0.7, 0.9, 1.0001],
                          labels=["<0.5", "0.5-0.7", "0.7-0.9", "≥0.9"], right=False)
res["lang_group"] = res["lang"].map(lambda l: l if l in ("es", "en") else "other (not measured)")

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
res["category"].value_counts().sort_values().plot.barh(ax=ax[0], title="Notes by category")
res["theme"].value_counts().sort_values().plot.barh(ax=ax[1], title="Notes by theme")
res["conf_band"].value_counts().sort_index().plot.bar(ax=ax[2], rot=0, title="Category confidence")
plt.tight_layout(); plt.show()

print("Share with category confidence ≥ 0.9 by language group:")
display(res.groupby("lang_group")["category_conf"].apply(lambda s: (s >= 0.9).mean()).round(3))
print("Category by language group (share):")
display(pd.crosstab(res["category"], res["lang_group"], normalize="columns").round(3))
print("Source:", res["source"].value_counts().to_dict())

## 7. Load into the database

Copy the file to the server and load it:

```bash
# on your computer (Google Drive for desktop syncs the file)
scp "~/Library/CloudStorage/GoogleDrive-<account>/My Drive/osm-notes-training/predictions/predictions_ft_20261109_open.csv.gz" \
    root@<server>:/root/osm-notes-deploy/output/

# on the server
cd ~/osm-notes-deploy && scripts/infer_load.sh output/predictions_ft_20261109_open.csv.gz
```

Then disconnect the runtime (**Runtime → Disconnect and delete runtime**) to stop
using compute units.